# 04 — Avaliação final no teste

A escolha das configurações é feita **somente pela acurácia de validação** e fica fixa antes de o teste ser tocado.

In [ ]:
# Setup: no Colab (site ou extensão do VS Code) clona o repositório do GitHub
# e instala as dependências; localmente só ajusta o sys.path.
# USE_DRIVE guarda pesos, JSONs e figuras no Google Drive para sobreviverem ao
# fim da sessão (o notebook 04 precisa dos pesos).
import os, sys
USE_DRIVE = True
# Detecta o Colab pela pasta /content em vez de "google.colab": no kernel da
# extensão do VS Code esse módulo pode não estar carregado.
IN_COLAB = os.path.isdir("/content") or "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/deep-vision-classification"
    # O código que roda é o do GitHub, não o da sua máquina: faça push antes.
    if not os.path.exists(REPO):
        !git clone -q https://github.com/davimoljo/deep-vision-classification.git {REPO}
    else:
        !git -C {REPO} pull -q
    %cd {REPO}
    !pip install -q -r requirements.txt
    if USE_DRIVE:
        if not os.path.isdir("/content/drive/MyDrive"):
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:
                print("Não montou o Drive:", e)
                print("No VS Code use 'Colab: Mount Google Drive to Server...' e rode esta célula de novo.")
        if os.path.isdir("/content/drive/MyDrive"):
            os.environ["OUTPUT_ROOT"] = "/content/drive/MyDrive/deep-vision-classification"
            print("Saídas em", os.environ["OUTPUT_ROOT"])
        else:
            print("ATENÇÃO: sem Drive, resultados e pesos somem quando o servidor desligar.")
    ROOT = REPO
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)

In [ ]:
from src.pipeline import (DEFAULTS, STRATEGIES, build_cnn, build_from_config,
                          build_pretrained, count_parameters, evaluate_on_test, fit,
                          get_loaders, list_results, load_best_weights, load_result,
                          result_exists)
from src.plots import comparison_table, per_class_accuracy, plot_confusion, plot_curves
DEFAULTS

In [ ]:
from collections import defaultdict

def best(runs):
    return max(runs, key=lambda r: load_result(r)["best_val_acc"])

cnn_runs = ["baseline"] + list_results("abl_")
# Runs "scratch" (sem ImageNet) são só para comparação, não concorrem à seleção.
tl_runs = [r for r in list_results("tl_") if not r.endswith("_scratch")]

# O enunciado pede o teste da melhor ESTRATÉGIA DE CADA arquitetura
# pré-treinada, não só da melhor rede de transfer no geral.
by_arch = defaultdict(list)
for run in tl_runs:
    by_arch[load_result(run)["model"]["arch"]].append(run)
best_per_arch = {arch: best(runs) for arch, runs in by_arch.items()}

best_cnn = best(cnn_runs)
# A baseline entra como referência; as demais são as vencedoras na validação.
FINAL = list(dict.fromkeys(["baseline", best_cnn, *best_per_arch.values()]))
print("melhor CNN própria:", best_cnn)
print("melhor estratégia por arquitetura:", best_per_arch)
FINAL

## Teste

In [ ]:
loaders_by_size = {}
for run in FINAL:
    r = load_result(run)
    size = r["hparams"]["img_size"]
    if size not in loaders_by_size:
        loaders_by_size[size] = get_loaders(r["hparams"]["batch_size"], img_size=size)
    model = load_best_weights(build_from_config(r["model"]), run)
    r = evaluate_on_test(model, loaders_by_size[size], r)
    print(f"{run}: val={r['best_val_acc']:.4f}  test={r['test_acc']:.4f}")

## Tabelas e figuras

In [ ]:
comparison_table(FINAL, name="final")

In [ ]:
comparison_table(cnn_runs + tl_runs, name="todas_runs").sort_values("Val acc (%)", ascending=False)

In [ ]:
for run in FINAL:
    plot_confusion(run)

In [ ]:
for run in FINAL:
    display(per_class_accuracy(run).style.set_caption(run))

In [ ]:
plot_curves(FINAL, name="curvas_finais");